In [ ]:
# Package imports

#import psi4
import psi4

# importing sympy for symbolic mantipulation
import sympy as sp

# import numpy for math and stuff
import numpy as np

# importing matplotlib module
from matplotlib import pyplot as plt

# for more sophisticated curve fitting
from scipy.optimize import curve_fit

## Background: The Morse Potential

Today you will continue your investigation of carbon monoxide using electronic structure calculations with psi4. Last week you learned how to calcuate the ground state energy for a single point (at a single specified bond length), in a specified basis set and at a specified level of theory.

This week you will:

1. Calculate a potential energy surface for CO.  This means you will calculate the energy at many different bond lengths.  You will use the MP2/cc-pvtz level of theory.  You should use example code from last weeks lab to acomplish this.

2. Use SciPy curve fitting to determine the dissociation energy $D_e$, the well width parameter $a$, and the equlibrium bond length $r_{eq}$ for the morse potential. 
$$
V(r) = D_e \big( 1 - e^{-a(r - r_{eq})} \big)^2
$$

3. The accepted bond dissociation energy for CO is 1072 kJ/mol and the accepted value of the bond length is 1.128 Angstroms, what are the percent errors you get from your curve fitting relative to the accepted values?

## Calculating a PES with Psi4

Recall our code for calculating the energy of an N$_2$ molecule last week. In this code we set the bond length for the N-N bond to 1.0.  To calculate a potential energy surface, we want to do the same type of calculation many times, changing the bond length.  This sounds like a job for a `for` loop!


In [ ]:
# Clean up before you start
psi4.core.clean()            # Clean temp files and scratch memory
psi4.core.clean_options()    # Reset all global options
psi4.core.clean_variables()

# Set your output file name
# The false option means you don't want the entire output to print
psi4.core.set_output_file('n2_calculations.out', False)

# Define a list of bond lengths for N2. 
bond_lengths = np.asarray([0.8, 0.9, 1.0, 1.1, 1.2, 1.3, 1.4, 1.5, 1.6, 1.7])
# Make an empty list to store the answers
mp2_energies = []

# Set up your molecule; we need to do the whole calculation for every bond length, 
# so everything is in the for loop
for bond in bond_lengths:
    n2 = psi4.geometry(f"""
    N
    N 1 {bond}
    """)

    # Set the basis set
    basis_set = 'cc-pvtz'
    psi4.set_options({'basis': basis_set})
    
    # Perform HF calculation
    mp2_energy = psi4.energy('mp2')

    # I am just printing so I can track the progress of my calculation.
    print(f'{bond}: {mp2_energy:.12f} Eh')
    # Here I am adding the energies to my list
    mp2_energies.append(mp2_energy)

In [ ]:
# Make a graph to look at the potential energy surface

plt.scatter(bond_lengths, mp2_energies)

## Gaussian Function Curve Fitting Example

This example demonstrates how to fit a Gaussian function to a set of data points using `scipy.optimize.curve_fit`. This method is widely used to model peaks in datasets across various scientific disciplines. You can apply the ideas from this example to the fitting of the Morse potential.

### Steps for Gaussian Fitting:

1. **Define the Gaussian function**: Specify the mathematical model for the Gaussian distribution.
2. **Prepare the data**: You need arrays of \( x \) values and their corresponding \( y \) values.
3. **Fit the Gaussian model to the data**: Use the `curve_fit` function to find the best fit parameters.
4. **Visualize the fit**: Plot the original data points along with the fitted curve to visually inspect the fit.

### Python Code for Fitting and Visualization

The following Python code block fits a Gaussian model to provided data and plots the results:


In [ ]:
# Define the Gaussian function
def gaussian(x, A, mu, sigma):
    return A * np.exp(-(x - mu)**2 / (2 * sigma**2))

# Example data
x = np.linspace(-5, 5, 100)
y = gaussian(x, 2.5, 0, 1) + 0.5 * np.random.normal(size=x.size)

# Initial guesses for fitting parameters
# You will need to change this to represnte parameters in the morse potential.
initial_guesses = [1, 0, 1]  # A, mu, sigma

# Perform the curve fit
params, params_covariance = curve_fit(gaussian, x, y, p0=initial_guesses)

# Extract the parameters
A_fit, mu_fit, sigma_fit = params
print(f"Fitted Amplitude (A):     {A_fit:12.12f}")
print(f"Fitted Mean (mu):         {mu_fit:12.12f}")
print(f"Fitted Std. Dev. (sigma): {sigma_fit:12.12f}")
print('\n\n')

#Calculate a bunch of points using the fitted equation
x_fit = np.linspace(-5, 5, 300)
y_fit = gaussian(x_fit, A_fit, mu_fit, sigma_fit)

# Plot the data and the fit on the same graph
plt.figure(figsize=(8, 5))
plt.scatter(x, y, label='Data')
plt.plot(x_fit, y_fit, color='red', label='Fitted Gaussian')
plt.title('Gaussian Fit to Data')
plt.xlabel('x')
plt.ylabel('y')
plt.legend()
plt.show()

## Part 1: Calculating the PES for CO

1.  Using the N$_2$ example code above, calculate a potential energy surface for CO using the MP2/cc-pvtz level of theory.  Using 30 bond lenghts starting at 0.8 angstroms and going to 1.8 angstroms.  In the end you should have a list or array of bond lengths and a list or array of your calculated energies.

## Part 2: Fitting your data to the Morse Potential
2.  The Morse potential describes the potential energy of a molecule when it is stretched or compressed from its equilibrium bond length.  At the equilbrium bond length, therefore, the potential energy should be zero.  To fit this structure, shift your energies values, such that the lowest energy you calculated is zero, and all the other values are shifted accordingly.
3.  Write a function that defines the Morse potential.  The inputs should be r, De, re, and a.
4.  Think about the physical meaning of the parameters, make reasonable guesses for De, re, and a.
5.  Use `scipy` to fit your data (bond lengths and the shifted energies) to the Morse potential.  Get the values of De, re, and a from the fit.
6.  Calculate a points for the fit curve and plot them on the same graph with your data.  **This is the graph you will turn in from today's lab.** Your caption should include the values of the fitted parameteres.
7.  The accepted bond dissociation energy for CO is 1072 kJ/mol and the accepted value of the bond length is 112.8 pm.  What are the percent errors you get from your curve fitting relative to the accepted values?  Be sure to pay attention to your units!


In [ ]:
# Clean up before you start
psi4.core.clean()            # Clean temp files and scratch memory
psi4.core.clean_options()    # Reset all global options
psi4.core.clean_variables()

# Your code goes here!